# Advanced tutorial: Explore storm response

## Research question

**How do streamflow and turbidity respond on unusually wet days at a STREAM station?**

## `stream_utils` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_water_quality()`
- `StreamClient.read_streamflow()`
- `StreamClient.read_meteorology()`
- `resample_timeseries()`
- `join_timeseries()`

The package intentionally does not define a universal storm event. This tutorial uses daily precipitation above the station's 95th percentile as a transparent example that researchers can replace.

In [ ]:
from stream_utils.client import StreamClient
from stream_utils.joins import join_timeseries
from stream_utils.temporal import resample_timeseries

stream = StreamClient()

## 1. Find a station and retrieve observations

In [ ]:
sites = stream.find_sites(variables="Turb_NTU", streamflow=True)
example_site = sites["STREAM_ID"].iloc[0]
start, end = "2019-01-01", "2021-12-31"
turbidity = stream.read_water_quality(
    example_site, "Turb_NTU", start, end, quality="approved"
)
streamflow = stream.read_streamflow(
    example_site, start, end, quality="approved"
)
rainfall = stream.read_meteorology(
    example_site, "APCP_surface", start, end
)

## 2. Create an aligned daily dataset

In [ ]:
daily_turbidity = resample_timeseries(turbidity, "Turb_NTU", "daily", "max")
daily_flow = resample_timeseries(streamflow, "Q_m3s", "daily", "max")
daily_rain = resample_timeseries(rainfall, "APCP_surface", "daily", "sum")
daily = join_timeseries(daily_turbidity, daily_flow)
daily = join_timeseries(daily, daily_rain)
daily.head()

## 3. Apply a researcher-defined storm threshold

In [ ]:
rain_threshold = daily["APCP_surface"].quantile(0.95)
storm_days = daily[daily["APCP_surface"] > rain_threshold].copy()
print(f"Threshold: {rain_threshold:.2f}")
print(f"Storm days: {len(storm_days):,}")
storm_days[["DateTime", "APCP_surface", "Q_m3s", "Turb_NTU"]].head(10)

In [ ]:
daily.assign(storm=daily["APCP_surface"] > rain_threshold).groupby("storm")[["Q_m3s", "Turb_NTU"]].median()